In [30]:
import os
import re
from datetime import datetime
from itertools import chain
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import spacy
from spacy.tokens.doc import Doc
from spacy.tokens.span import Span
from spacy.tokens.token import Token
import coreferee
from coreferee.data_model import ChainHolder
from helpers import get_request
from bs4 import BeautifulSoup
from bs4.element import Tag

In [2]:
nlp = spacy.load("en_core_web_lg")
nlp.add_pipe("coreferee")

In [3]:
def get_text_from_file(path: str):
    text = ""
    
    try:
        with open(path, "r", encoding="utf8") as file:
            text = file.read()
    except Exception as e:
        print("Error reading file", path)
        print(e)
    finally:
        return text

In [4]:
articles_folder = "./articles"

In [5]:
article_files = os.listdir(articles_folder)
article_files = [file for file in article_files if file.endswith(".txt")]

In [6]:
texts = [get_text_from_file(f"{articles_folder}/{file}") for file in article_files]

df = pd.DataFrame(data={
    "file": article_files,
    "text": texts
})
df

,file,text
0,news_10-get-ready-for-a-november-to-remember-i...,"November is stacked. Normally, I would look to..."
1,news_10-june-schedule-jam-packed-goodness.txt,"Buckle up, kids, because the June schedule is ..."
2,news_10-massive-fights-schedule-may.txt,There are all kinds of different ways to look ...
3,news_10-memorable-moments-down-under.txt,"On Saturday, February 8, the UFC returns to Qu..."
4,news_10-middleweights-biggest-championship-mom...,Fourteen individuals have held the UFC middlew...
...,...,...
761,news_zhang-mingyang-all-gas-no-brakes-ufc-kans...,Rising Chinese star Zhang Mingyang has a stat ...
762,news_zhang-mingyang-embracing-spotlight-ufc-sh...,Zhang Mingyang has all the makings of a potent...
763,news_zhang-mingyang-journey-main-event-light-h...,Zhang Mingyang first entered the UFC ecosystem...
764,news_zhang-weili-builds-her-case-goat-status.txt,


In [7]:
def already_split_gender_paragraphs(article_file: str):
    gender_paragraphs_folder = f"{articles_folder}/{article_file.replace('.txt', '')}"
    return os.path.exists(gender_paragraphs_folder)

In [8]:
# Since this step is very resource intensive, we only want to
# run it on articles that haven't already been split on gender
df_to_separate = df[~df["file"].apply(already_split_gender_paragraphs)].copy()
df_to_separate

,file,text
0,news_10-get-ready-for-a-november-to-remember-i...,"November is stacked. Normally, I would look to..."
14,news_adrian-yanez-reloaded-ufc-vegas-111.txt,The first two years of Adrian Yanez’ UFC caree...
30,news_alice-ardelean-is-planning-her-big-move-u...,Alice Ardelean remains close to her Romanian r...
43,news_ante-delija-making-waves.txt,Making an immediate impact in the UFC is a cha...
67,news_bigger-picture-ufc-321-aspinall-vs-gane.txt,The UFC’s annual October pay-per-view event in...
110,news_chris-padilla-blocking-out-the-noise-ufc-...,"For Chris Padilla, it doesn’t matter where he ..."
111,news_christian-leroy-duncan-settled-and-starti...,Here’s a very simple truth that doesn’t get ac...
154,news_david-onama-excited-but-also-expected-veg...,David Onama hadn’t been to sign posters for th...
163,news_donte-johnson-clocking-in-ufc-vegas-110.txt,Donte Johnson claimed his place on the UFC ros...
205,news_fight-by-fight-preview-ufc-321-aspinall-v...,The UFC makes its second trip to Abu Dhabi thi...


In [9]:
# This takes a long time, so have patience
df_to_separate["doc"] = list(nlp.pipe(df_to_separate["text"]))

In [10]:
def get_people(doc: Doc):
    return [ent for ent in doc.ents if ent.label_ == "PERSON"]

In [11]:
df_to_separate["people"] = df_to_separate["doc"].apply(get_people)

In [45]:
all_people: list[Span] = list(chain.from_iterable(df_to_separate["people"]))
all_names = set(p.text for p in all_people)
full_names = set(name for name in all_names 
                 if 
                 len(name.split()) >= 2 
                 and len(name.split()) <= 3
                 and re.search("^[a-zA-Z\u00C0-\u017F\s’]+$", name) is not None)

full_names

{'Addison Barger’s',
 'Adrian Yanez',
 'Adrian Yanez’ UFC',
 'Aiemann Zahabi',
 'Aleksandar Rakic',
 'Aleksandar Rakić',
 'Alex Pereira',
 'Alexa Grasso',
 'Alexander Volkov',
 'Alexandre Pantoja',
 'Alice Ardelean',
 'Allan Nascimento',
 'Amanda Lemos',
 'Amanda Nunes',
 'Amanda Ribas',
 'Anderson Silva',
 'Andre Lima',
 'Andre Muniz',
 'Andreas Gustafsson',
 'Andrei Arlovski',
 'Andrey Pulyaev',
 'Ange Loosa',
 'Angela Hill',
 'Ante Delija',
 'Ariane Carnelossi',
 'Arman Tsarukyan',
 'Aspen Ladd',
 'Asu Almabayev',
 'Ateba Gautier',
 'Austin Vanderford',
 'Azamat Murzakanov',
 'Azat Maksum',
 'Bantamweights Miles Johns',
 'Battering Aldana',
 'Belal Muhammad',
 'Benoît Saint Denis',
 'Bia Mesquita',
 'Billy Elekana',
 'Brazilian Ketlen Vieira',
 'Brendon Marotte',
 'Bruna Brasil',
 'Brunno Ferreira',
 'Bryan Battle',
 'Bueno Silva',
 'Calvin Kattar',
 'Cameron Saaiman',
 'Cameron Smotherman',
 'Carla Esparza',
 'Carlos Hernandez',
 'Carlos Leal',
 'Carlos Prates',
 'Carlos Ulberg',
 

In [14]:
def is_woman_tag(tag: Tag):
    text = tag.get_text().strip().lower()
    return "woman" in text or "women" in text

def find_gender_on_ufc(name: str):
    name_query = "-".join(name.split()).replace('’', ' ')
    url = f"https://www.ufc.com/athlete/{name_query}"

    try:
        response = get_request(url)
        soup = BeautifulSoup(response.text, "html.parser")

        tags: list[Tag] = soup.find_all("p", "hero-profile__tag")
        if not tags:
            return None

        is_woman = any([is_woman_tag(tag) for tag in tags])
        if is_woman:
            return "woman"
        else:
            return "man"
    except Exception as e:
        print("Error getting gender of", name)
        print(e)

In [ ]:
# Set this to False if you want to fetch the genders from the UFC website
use_saved_gender_map = False
genders_map_file = "./genders_map.csv"

if use_saved_gender_map and os.path.exists(genders_map_file):
    name_gender_pairs_df = pd.read_csv(genders_map_file)
    genders_map: dict[str, str] = dict(zip(name_gender_pairs_df["name"], name_gender_pairs_df["gender"]))
else:
    def get_name_gender_pair_on_ufc(name: str):
        return (name, find_gender_on_ufc(name))

    with ThreadPoolExecutor() as executor:
        name_gender_pairs = [
            (n, g)
            for n, g in executor.map(get_name_gender_pair_on_ufc, full_names)
            if g is not None
        ]

    genders_map = dict(name_gender_pairs)

    for name, gender in name_gender_pairs:
        for name_component in name.split():
            genders_map[name_component] = gender

    pd.DataFrame(data={
        "name": genders_map.keys(),
        "gender": genders_map.values(),
    })\
        .set_index("name")\
        .to_csv(f"./genders_map_{datetime.now().strftime('%d-%m-%y_%H-%M-%S')}.csv")

genders_map

{'Azamat Murzakanov': 'man',
 'Magomed Ankalaev': 'man',
 'Norma Dumont': 'woman',
 'Jared Gordon': 'man',
 'Michael Morales': 'man',
 'Khamzat Chimaev': 'man',
 'Edson Barboza': 'man',
 'Austin Vanderford': 'man',
 'Serghei Spivac': 'man',
 'Rhys McKee': 'man',
 'Azat Maksum': 'man',
 'Michael Chiesa': 'man',
 'Jacqueline Cavalcanti': 'woman',
 'Randy Brown': 'man',
 'Steve Garcia': 'man',
 'Zhang Weili': 'woman',
 'Hyder Amil': 'man',
 'Mitch Raposo': 'man',
 'Roberto Romero': 'man',
 'Melissa Martinez': 'woman',
 'Yadier del Valle': 'man',
 'Karol Rosa': 'woman',
 'Garrett Armfield': 'man',
 'Aleksandar Rakic': 'man',
 'Conor McGregor': 'man',
 'Stephen Thompson': 'man',
 'Deiveson Figueiredo': 'man',
 'Kyoji Horiguchi': 'man',
 'Billy Elekana': 'man',
 'Felice Herrig': 'woman',
 'Mateusz Gamrot': 'man',
 'Josh Hokit': 'man',
 'Pete Rodriguez': 'man',
 'Francis Marshall': 'man',
 'Khaos Williams': 'man',
 'Ryan Spann': 'man',
 'Daniel Marcos': 'man',
 'Bruna Brasil': 'woman',
 'Rafa

In [16]:
def get_coreference_genders(row: pd.Series):
    doc: Doc = row["doc"]

    chains: ChainHolder = doc._.coref_chains
    
    def loop():
        for chain in chains:
            main_item_mention = chain[chain.most_specific_mention_index]
            main_item = doc[main_item_mention.root_index]
            
            if main_item.text in genders_map:
                for mention in chain:
                    token = doc[mention.root_index]
                    yield (token, genders_map[main_item.text])
    
    return list(loop())

In [17]:
df_to_separate["coreference_genders"] = df_to_separate.apply(get_coreference_genders, axis="columns")

In [18]:
def get_people_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    people: list[Span] = row["people"]
    coreference_genders: list[tuple[Token, str]] = row["coreference_genders"]

    people_with_gender = [p for p in people if genders_map.get(p.text) == gender]
    coreferences_with_gender = [p for p, g in coreference_genders if g == gender]
    coreferences_with_gender = [Span(doc, p.i, p.i + 1) for p in coreferences_with_gender]
    
    return set(people_with_gender + coreferences_with_gender)

In [19]:
df_to_separate["men"] = df_to_separate.apply(get_people_with_gender, args=("man",), axis="columns")
df_to_separate["women"] = df_to_separate.apply(get_people_with_gender, args=("woman",), axis="columns")

In [20]:
def get_paragraph_boundaries(doc: Doc):
    line_breaks = [token for token in doc if token.text == "\n"]
    line_break_indexes = [lb.i for lb in line_breaks]
    paragraph_boundaries = list(zip([0] + line_break_indexes, line_break_indexes + [len(doc)]))
    return paragraph_boundaries

In [21]:
df_to_separate["paragraph_boundaries"] = df_to_separate["doc"].apply(get_paragraph_boundaries)

In [22]:
def count_genders_in_paragraphs(row: pd.Series, gender: str):
    people_with_gender: set[Span] = row[gender]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]

    def count_gender_in_paragraph(boundary: tuple[int, int]):
        start, end = boundary
        count = 0

        for person in people_with_gender:
            if person.end >= end:
                continue
            elif person.start < start:
                continue
            
            count += 1

        return count

    return [count_gender_in_paragraph(boundary) for boundary in paragraph_boundaries]

In [23]:
df_to_separate["men_per_paragraph"] = df_to_separate.apply(count_genders_in_paragraphs, args=("men",), axis="columns")
df_to_separate["women_per_paragraph"] = df_to_separate.apply(count_genders_in_paragraphs, args=("women",), axis="columns")

In [24]:
def assign_gender_to_paragraphs(row: pd.Series):
    men_per_paragraph: list[int] = row["men_per_paragraph"]
    women_per_paragraph: list[int] = row["women_per_paragraph"]

    def choose_gender(m: int, f: int):
        if m == 0 and f == 0:
            return None
        elif m > f:
            return "man"
        elif f > m:
            return "woman"
        else: 
            return "equal"

    return [choose_gender(m, f) for m, f in zip(men_per_paragraph, women_per_paragraph)]

In [25]:
df_to_separate["paragraph_genders"] = df_to_separate.apply(assign_gender_to_paragraphs, axis="columns")

In [26]:
def get_paragraphs_with_gender(row: pd.Series, gender: str):
    doc: Doc = row["doc"]
    paragraph_boundaries: list[tuple[int, int]] = row["paragraph_boundaries"]
    paragraph_genders: list[str | None] = row["paragraph_genders"]

    paragraphs_indexes_with_gender = [i for i, g in enumerate(paragraph_genders) if g == gender]
    paragraphs_boundaries_with_gender = [paragraph_boundaries[i] for i in paragraphs_indexes_with_gender]
    paragraphs_with_gender = [doc[start:end] for start, end in paragraphs_boundaries_with_gender]

    return paragraphs_with_gender

In [27]:
df_to_separate["man_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("man",), axis="columns")
df_to_separate["woman_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("woman",), axis="columns")
df_to_separate["equal_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=("equal",), axis="columns")
df_to_separate["genderless_paragraphs"] = df_to_separate.apply(get_paragraphs_with_gender, args=(None,), axis="columns")
df_to_separate

,file,text,doc,people,coreference_genders,men,women,paragraph_boundaries,men_per_paragraph,women_per_paragraph,paragraph_genders,man_paragraphs,woman_paragraphs,equal_paragraphs,genderless_paragraphs
0,news_10-get-ready-for-a-november-to-remember-i...,"November is stacked. Normally, I would look to...","(November, is, stacked, ., Normally, ,, I, wou...","[(Ketlen, Vieira), (Norma, Dumont), (Vieira), ...","[(Vieira, woman), (Vieira, woman), (her, woman...","{(his), (Nikolay, Veretennikov), (Makhachev), ...","{(Namajunas), (Kayla, Harrison), (Namajunas), ...","[(0, 67), (67, 103), (103, 112), (112, 153), (...","[0, 0, 0, 0, 0, 0, 0, 5, 8, 10, 0, 3, 6, 10, 0...","[0, 0, 0, 4, 11, 9, 3, 0, 0, 0, 0, 0, 0, 0, 0,...","[None, None, None, woman, woman, woman, woman,...","[(\n, Steve, Garcia, and, David, Onama, clash,...","[(\n, You, know, it, ’s, a, loaded, month, whe...",[],"[(November, is, stacked, ., Normally, ,, I, wo..."
14,news_adrian-yanez-reloaded-ufc-vegas-111.txt,The first two years of Adrian Yanez’ UFC caree...,"(The, first, two, years, of, Adrian, Yanez, ’,...","[(Adrian, Yanez, ’, UFC), (Dana, White, ’s), (...","[(Yanez, man), (Yanez, man), (it, man), (Marco...","{(he), (Yanez), (Yanez), (Daniel, Marcos), (Qu...",{},"[(0, 79), (79, 128), (128, 170), (170, 250), (...","[0, 0, 0, 2, 3, 2, 0, 8, 3, 1, 1, 2, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[None, None, None, man, man, man, None, man, m...","[(\n, “, I, never, thought, something, like, t...",[],[],"[(The, first, two, years, of, Adrian, Yanez, ’..."
30,news_alice-ardelean-is-planning-her-big-move-u...,Alice Ardelean remains close to her Romanian r...,"(Alice, Ardelean, remains, close, to, her, Rom...","[(Alice, Ardelean), (Ardelean), (Ardelean), (G...","[(Ardelean, woman), (her, woman), (herself, wo...",{(Garcia)},"{(Melissa, Martinez), (she), (Ardelean), (Arde...","[(0, 25), (25, 67), (67, 175), (175, 210), (21...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[4, 5, 3, 3, 6, 7, 2, 4, 1, 6, 1, 0, 6, 5, 3, ...","[woman, woman, woman, woman, woman, woman, wom...",[],"[(Alice, Ardelean, remains, close, to, her, Ro...",[],"[(\n, “, I, had, an, even, tougher, fight, cam..."
43,news_ante-delija-making-waves.txt,Making an immediate impact in the UFC is a cha...,"(Making, an, immediate, impact, in, the, UFC, ...","[(Ante, Delija), (Marcin, Tybura), (Waldo, Cor...","[(Delija, man), (Delija, man), (Tybura, man), ...","{(his), (Aspinall), (Delija), (Delija), (his),...",{},"[(0, 67), (67, 133), (133, 207), (207, 245), (...","[0, 4, 5, 1, 0, 5, 2, 1, 2, 1, 0, 0, 6, 5, 6, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[None, man, man, man, None, man, man, man, man...","[(\n, But, things, are, a, little, different, ...",[],[],"[(Making, an, immediate, impact, in, the, UFC,..."
67,news_bigger-picture-ufc-321-aspinall-vs-gane.txt,The UFC’s annual October pay-per-view event in...,"(The, UFC, ’s, annual, October, pay, -, per, -...","[(Tom, Aspinall), (Ciryl, Gane), (Jon, Anik), ...","[(Aspinall, man), (himself, man), (he, man), (...","{(Rakic), (he), (Alex, Pereira), (himself), (M...","{(Mackenzie, Dern), (she), (she), (Dern), (Der...","[(0, 59), (59, 67), (67, 135), (135, 255), (25...","[0, 0, 2, 2, 0, 4, 0, 5, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 0, 0, 3, 3, ...","[None, None, man, man, None, man, None, man, N...","[(\n, The, way, Saturday, ’s, highly, anticipa...","[(\n, Mackenzie, Dern, became, the, sixth, wom...",[],"[(The, UFC, ’s, annual, October, pay, -, per, ..."
110,news_chris-padilla-blocking-out-the-noise-ufc-...,"For Chris Padilla, it doesn’t matter where he ...","(For, Chris, Padilla, ,, it, does, n’t, matter...","[(Chris, Padilla), (Padilla), (Jai, Herbert), ...","[(Padilla, man), (he, man), (He, man), (Padill...","{(his), (Padilla), (He), (Padilla), (Chris, Pa...",{},"[(0, 47), (47, 123), (123, 221), (221, 258), (...","[6, 6, 9, 5, 7, 6, 4, 8, 1, 3, 3, 3]","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[man, man

In [28]:
def save_separate_paragraphs(row: pd.Series):
    original_file: str = row["file"]
    new_folder = original_file.replace(".txt", "")

    os.makedirs(f"./articles/{new_folder}", exist_ok=True)

    man_paragraphs: list[Span] = row["man_paragraphs"]
    man_text = "\n".join([p.text for p in man_paragraphs])

    with open(f"./articles/{new_folder}/man.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(man_text)

    woman_paragraphs: list[Span] = row["woman_paragraphs"]
    woman_text = "\n".join([p.text for p in woman_paragraphs])

    with open(f"./articles/{new_folder}/woman.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(woman_text)

    equal_paragraphs: list[Span] = row["equal_paragraphs"]
    equal_text = "\n".join([p.text for p in equal_paragraphs])

    with open(f"./articles/{new_folder}/equal.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(equal_text)

    genderless_paragraphs: list[Span] = row["genderless_paragraphs"]
    genderless_text = "\n".join([p.text for p in genderless_paragraphs])

    with open(f"./articles/{new_folder}/genderless.txt", "w", encoding="utf8", newline="\n") as file:
        file.write(genderless_text)

In [29]:
df_to_separate.apply(save_separate_paragraphs, axis="columns")

0      None
14     None
30     None
43     None
67     None
110    None
111    None
154    None
163    None
205    None
208    None
223    None
241    None
257    None
280    None
359    None
371    None
387    None
460    None
495    None
533    None
552    None
570    None
615    None
650    None
659    None
676    None
684    None
700    None
701    None
713    None
714    None
715    None
716    None
726    None
734    None
753    None
dtype: object